# Predicting Midprice Direction from Limit Order Book Data

I compare linear logistic regression, logistic regression with quadratic features, and XGBoost to predict whether the next midprice move is up or down. My inputs are prices and volumes from the first four levels of the order book, together with the five previous midprice change directions.

## Dataset setup

*The dataset description and loading instructions in this opening section are adapted from the original course material.*

The following cell downloads and extracts the course data archive. The two files used in this project are `data_full/Data_A.csv` and `data_full/Data_B.csv`.

The download cell requires `wget` and `unzip`. If the archive has already been extracted into `data_full`, this cell can be skipped. The rest of the notebook uses Python with NumPy, pandas, Matplotlib, scikit-learn and XGBoost.

In [16]:
import os

dropbox_url = "https://www.dropbox.com/scl/fi/e1iara9nw3dr4fjm03avy/data_full.zip?rlkey=w4gb879dfx06qjygr5imbplmj&st=9rl20ukp&dl=1"
zip_file_name = "data_full.zip"

print(f"Downloading {zip_file_name}...")

# Download the file using wget
!wget -q -O {zip_file_name} "{dropbox_url}"

print("Download complete. Unzipping...")

# Unzip the file quietly
!unzip -q {zip_file_name}
os.remove(zip_file_name)

print("\nUnzipping complete. Contents of the 'data_full' folder:")

# List the contents of the newly created 'data_full' directory
!ls data_full/

Download complete. Unzipping...
replace data_full/Data_A.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
Unzipping complete. Contents of the 'data_full' folder:
 Data_A.csv			       sp500_prices_2010_2024.csv
 Data_B.csv			       sp500stocks.csv
 F-F_Momentum_Factor_daily.csv	      '^SP500TR.csv'
 F-F_Research_Data_Factors_daily.csv   sp500_universe_2010_2024.csv
 gspc.csv


### Dataset structure

`Data_A.csv` contains 100,000 observations and 22 columns, with no header row.

The first column is the direction of the next midprice change: **0 = down**, **1 = up**. The midprice is defined using the best bid and ask:

$$
\text{midprice} = \frac{\text{best bid price} + \text{best ask price}}{2}.
$$

The remaining 21 columns contain features recorded just before the midprice change associated with the label. Prices are recorded in US dollars multiplied by 10,000; volumes are numbers of shares.

| Column(s), counting from 1 | Description | Column name(s) used below |
| --- | --- | --- |
| 1 | Next midprice change direction | `label` |
| 2 | Level 1 ask price | `askl1` |
| 3 | Level 1 ask volume | `vola1` |
| 4 | Level 1 bid price | `bidl1` |
| 5 | Level 1 bid volume | `volb1` |
| 6 | Level 2 ask price | `askl2` |
| 7 | Level 2 ask volume | `vola2` |
| 8 | Level 2 bid price | `bidl2` |
| 9 | Level 2 bid volume | `volb2` |
| 10 | Level 3 ask price | `askl3` |
| 11 | Level 3 ask volume | `vola3` |
| 12 | Level 3 bid price | `bidl3` |
| 13 | Level 3 bid volume | `volb3` |
| 14 | Level 4 ask price | `askl4` |
| 15 | Level 4 ask volume | `vola4` |
| 16 | Level 4 bid price | `bidl4` |
| 17 | Level 4 bid volume | `volb4` |
| 18–22 | Five previous midprice change directions, coded 0 or 1 | `m1`, `m2`, `m3`, `m4`, `m5` |

According to the course dataset notes, the rows were randomly drawn from a larger dataset covering **1 August–30 October 2023** and are treated as independent samples in this exercise. The original time-series order cannot be recovered from the supplied files.

`Data_B.csv` contains 10,000 observations sampled in the same way, with the same 22-column layout. In the supplied version, its first column includes labels so that test accuracy can be calculated. It is used as a separate test set for the two logistic regression models.

## Research approach

I start with logistic regression as a baseline, then add quadratic terms to capture nonlinear relationships and interactions. I also fit XGBoost to explore whether boosted trees improve validation accuracy.

I use 80,000 rows from `Data_A` for training and the remaining 20,000 for validation. I calculate all scaling statistics on the training split. I use `Data_B` for a separate test comparison of the two logistic regression models; the XGBoost results below cover training and validation only.

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

## 1. Inspecting the data

I give the columns descriptive names, inspect their types and missing-value counts, and check the balance between upward and downward moves.

In [18]:
data_dir = Path('data_full')

In [19]:
file_path = data_dir / 'Data_A.csv'
data = pd.read_csv(file_path, header=None)

data.columns = [
    "label", "askl1", "vola1", "bidl1", "volb1", "askl2",
    "vola2", "bidl2", "volb2", "askl3", "vola3", "bidl3", "volb3",
    "askl4", "vola4", "bidl4", "volb4",
    "m1", "m2", "m3", "m4", "m5"
]

display(data.head())
print(data.info())

,label,askl1,vola1,bidl1,volb1,askl2,vola2,bidl2,volb2,askl3,...,volb3,askl4,vola4,bidl4,volb4,m1,m2,m3,m4,m5
0,1,428900.0,1,428700.0,200,429000.0,100,428500.0,300,429100.0,...,300,429200.0,200,428300.0,100,0,1,0,1,0
1,1,427100.0,100,427000.0,100,427200.0,940,426900.0,100,427300.0,...,100,427400.0,700,426700.0,100,0,1,0,1,0
2,1,511300.0,100,511200.0,129,511400.0,500,511100.0,200,511500.0,...,300,511600.0,200,510900.0,300,0,1,1,0,0
3,0,415600.0,200,415400.0,100,415700.0,100,415300.0,100,415800.0,...,600,415900.0,300,415100.0,1020,0,1,1,0,1
4,0,506600.0,300,506500.0,100,506700.0,100,506300.0,200,506800.0,...,1099,506900.0,214,506100.0,699,0,0,1,1,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   label   100000 non-null  int64  
 1   askl1   100000 non-null  float64
 2   vola1   100000 non-null  int64  
 3   bidl1   100000 non-null  float64
 4   volb1   100000 non-null  int64  
 5   askl2   100000 non-null  float64
 6   vola2   100000 non-null  int64  
 7   bidl2   100000 non-null  float64
 8   volb2   100000 non-null  int64  
 9   askl3   100000 non-null  float64
 10  vola3   100000 non-null  int64  
 11  bidl3   100000 non-null  float64
 12  volb3   100000 non-null  int64  
 13  askl4   100000 non-null  float64
 14  vola4   100000 non-null  int64  
 15  bidl4   100000 non-null  float64
 16  volb4   100000 non-null  int64  
 17  m1      100000 non-null  int64  
 18  m2      100000 non-null  int64  
 19  m3      100000 non-null  int64  
 20  m4      100000 non-null  int64  
 21  m5      100

In [20]:
# Counts both classes to check whether one direction dominates the dataset.
occ = data['label'].value_counts()
occ

,count
label,
1,50267
0,49733


In [21]:
# Expresses the counts as proportions to make the class balance easier to interpret.
ratio_cases = occ / len(data.index)
print(f'Ratio of upward movement cases: {ratio_cases[1]}\nRatio of downward movement cases: {ratio_cases[0]}')

Ratio of upward movement cases: 0.50267
Ratio of downward movement cases: 0.49733


## 2. Training and validation split

Upward moves account for approximately 50.27% of `Data_A`, so the classes are close to balanced. This gives me a useful reference when interpreting accuracy: always predicting the more common class would score about 50.27% on the full dataset.

I separate the label from the features and reserve the last 20,000 rows for validation. Because the supplied observations were randomly sampled, this is a split by row position rather than a chronological split.

In [22]:
# Keeps the target separate so it cannot be used as an input feature.
x_train_full = data.drop("label", axis=1)
y_train_full = np.array(data["label"])

# Uses the first 80,000 sampled rows for fitting and the remaining 20,000 for validation.
split_ind = 80000
x_train = x_train_full[:split_ind]
y_train = y_train_full[:split_ind]
x_val = x_train_full[split_ind:]
y_val = y_train_full[split_ind:]

### Feature scaling

Prices and volumes have very different scales. I standardise each feature by subtracting its training mean and dividing by its training standard deviation. This puts the inputs on a comparable scale for logistic regression.

I apply those same training statistics to the validation data, so the validation observations do not influence the scaling.

In [23]:
# Calculates the scaling statistics using only the training observations.
means = x_train.mean(0)
stds = x_train.std(0)

# Reuses those statistics for validation to avoid leaking information into preprocessing.
x_train = (x_train - means) / stds
x_val = (x_val - means) / stds

## 3. Linear logistic regression

I first fit logistic regression to the 21 standardised features. This gives me a baseline with a linear decision boundary before I introduce nonlinear features or a more flexible model.

In [24]:
import sklearn
from sklearn.linear_model import LogisticRegression

# Fits a linear baseline before adding interactions or trying boosted trees.
reg = LogisticRegression(fit_intercept=True).fit(x_train, y_train)

print('Train Accuracy: ', reg.score(x_train, y_train))
print('Validation Accuracy: ', reg.score(x_val, y_val))

Train Accuracy:  0.7195625
Validation Accuracy:  0.71935


## 4. Logistic regression with quadratic features

The linear model reaches **71.94% validation accuracy**, close to its **71.96% training accuracy**.

I next expand the inputs to include squared terms and pairwise interactions. The classifier is still logistic regression, but these additional features let it represent a nonlinear decision boundary in the original inputs.

In [25]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Adds squares and pairwise interactions before fitting logistic regression.
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=True),
    LogisticRegression(fit_intercept=True, solver='newton-cg')
)

poly_model.fit(x_train, y_train)

# Compares both scores to see whether the extra flexibility helps on validation data.
print('Train Accuracy: ', poly_model.score(x_train, y_train))
print('Validation Accuracy: ', poly_model.score(x_val, y_val))

Train Accuracy:  0.735725
Validation Accuracy:  0.73335


Validation accuracy increases to **73.34%**, compared with **71.94%** for the linear baseline. Training accuracy is **73.57%**, so the gap between training and validation remains small. This supports testing the quadratic model on the separate `Data_B` sample.

## 5. Test results for the logistic regression models

I load `Data_B` and apply the scaling statistics from the training split. I then evaluate both fitted logistic regression models against its labels.

In [ ]:
# Loads the separate test sample with the same column layout as the training data.
file_path = data_dir / 'Data_B.csv'
df_test = pd.read_csv(file_path, header=None)
df_test.columns = [
    "label", "askl1", "vola1", "bidl1", "volb1", "askl2",
    "vola2", "bidl2", "volb2", "askl3", "vola3", "bidl3", "volb3",
    "askl4", "vola4", "bidl4", "volb4",
    "m1", "m2", "m3", "m4", "m5"
]

# Keeps these labels for evaluation.a
x_test = df_test.drop("label", axis=1)
y_test = np.array(df_test["label"])

# Standardising.
x_test = (x_test - means) / stds

In [ ]:
# Compares the fitted models on the same separate test sample.
print('Test Accuracy of Linear Model: ', reg.score(x_test, y_test))
print('Test Accuracy of Quadratic Model: ', poly_model.score(x_test, y_test))

The linear model scores **71.69%** on `Data_B`, while the quadratic model scores **73.38%**. The improvement from quadratic features therefore carries over to this separate sample, with a gain of **1.69 percentage points** in test accuracy.

## 6. XGBoost extension

I also fit XGBoost to capture nonlinear relationships without explicitly constructing polynomial features. It builds an ensemble of trees, with each new tree working to reduce the loss of the current model.

I limit the trees to depth 3 and use a learning rate of 0.05. Row and feature subsampling, together with L2 regularisation, constrain the model. I allow up to 1,000 trees and use validation log loss for early stopping, with a patience of 30 rounds.

I reuse the same training and validation inputs for consistency. Tree models do not require standardisation, but this keeps the preprocessing shared across the comparison. Because I use validation loss to select the best boosting iteration, this validation score is part of model selection rather than an untouched test result.

In [ ]:
from xgboost import XGBClassifier

# Shallow trees and small boosting steps to control the model's flexibility.
xgb_model = XGBClassifier(
    objective="binary:logistic",  # probability of an upward move.
    n_estimators=1000,  # up to 1,000 boosting rounds.
    max_depth=3,
    learning_rate=0.05,
    subsample=0.80,  # sample 80% of the training rows for each tree.
    colsample_bytree=0.80,  # sample 80% of the features for each tree.
    reg_lambda=1.0,  # L2 penalty to the leaf scores.
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=30,  # stops if validation loss does not improve for 30 rounds.
    random_state=42,
    n_jobs=4
)

# Fits on the training split and use validation loss to select the best iteration.
xgb_model.fit(
    x_train,
    y_train,
    eval_set=[(x_val, y_val)],
    verbose=False
)

In [29]:
print(f"Train accuracy: {xgb_model.score(x_train, y_train):.4%}")
print(f"Validation accuracy: {xgb_model.score(x_val, y_val):.4%}")
print(f"Trees used for prediction: {xgb_model.best_iteration + 1}")
print(f"Test accuracy of XGBoost: {xgb_model.score(x_test, y_test):.4%}")

Train accuracy: 78.7613%
Validation accuracy: 77.5000%
Trees used for prediction: 994
Test accuracy of XGBoost: 77.6100%


## 7. Results and interpretation

The saved outputs give the following comparison:

| Model | Training accuracy | Validation accuracy | Test accuracy (`Data_B`) |
| --- | ---: | ---: | ---: |
| Linear logistic regression | 71.96% | 71.94% | 71.69% |
| Logistic regression with quadratic features | 73.57% | 73.34% | 73.38% |
| XGBoost | 78.76% | 77.50% | 77.61% |

Quadratic features improve both validation and test accuracy over the linear baseline. XGBoost achieves the highest accuracy on both sets, reaching **77.50% on validation** and **77.61% on the test set**, using **994 trees** at the best validation-loss iteration.

Compared with the quadratic model, XGBoost improves test accuracy by **4.23 percentage points**. Its test accuracy is also close to its validation accuracy, suggesting that the improvement carries over to the separate sample. The gap between training and test accuracy is approximately **1.15 percentage points**.

I used validation log loss to select the best boosting iteration, then evaluated the fitted model on `Data_B`. The test set was not used for fitting or early stopping.

### Limitations and next steps

I interpret these scores as evidence about classification on the supplied samples. They do not measure trading profitability: this notebook does not model execution, spreads, fees or price impact. Since the original time order is unavailable, I also cannot use these files for a chronological backtest.

My next steps would be to inspect probability calibration and errors by class, investigate which order book features contribute most to the predictions, and repeat the analysis on timestamped data with chronological splits before assessing a trading strategy.